# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Maame Aba Kwofie
**Student ID:** 32952028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",  # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"  # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response


#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("How do I know if I'm on the right path, academically?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

Knowing if you're on the right academic path can be a challenging and personal question. Here are some signs that may indicate you're on the right track:

1. **Alignment with your interests**: Are you studying something that genuinely fascinates you? Do you enjoy learning about the subject matter and look forward to exploring it further?
2. **Career goals**: Does your academic path align with your career aspirations? Are you gaining the skills and knowledge required for your desired profession?
3. **Personal growth**: Are you developing new skills, such as critical thinking, problem-solving, or communication? Are you becoming a more confident, independent, and self-motivated individual?
4. **Challenging but manageable**: Are you being challenged academically, but still able to manage your coursework and responsibilities? A sense of accomplishment and progress can be a good indicator that you're on the right path.
5. **Supportive environment**: Are you surrounded by supportive peers, me

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. 
    The system role is the basis for the overall behaviour of the LLM, such as its personna, the context in which its acting, and rules and contraints for the output. For example, You are a high school counselor. Provide concise advice using high-school level vocabulary

    The user role contains the specific question or prompt given by the user. For example, "How do I juggle my academics with my extra-cirriculars"

2. 
    A token is the atomic unit a model reads. LLMs compute predictions for each token sequentially. A request that requires/asks for a 1-word response takes significantly less time and resources than a prompt that requires a 100-word response. Billing per token ensures users are charged proportionally to the actual workload generated.

### Part 1.2 — Temperature: the randomness dial

In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
print("Temperature = 0.0")
for i in range(1, 6):
    res = ask_llm(test_question, temperature=0.0)
    print(f"Run {i}: {res.choices[0].message.content.strip()}\n")

print("Temperature = 1.2")
for i in range(1, 6):
    res = ask_llm(test_question, temperature=1.2)
    print(f"Run {i}: {res.choices[0].message.content.strip()}\n")


Temperature = 0.0
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word that means "honest" or "transparent," which could help build trust with potential customers.
7. **Traders' Fund**: This name is straightforward and emph

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

At Temperature = 0.0: The outputs were very repetitive throughout all 5 runs. The model repeatedly selected the exact same set of suggestions and maintained nearly identical structure and phrasing across runs.

At Temperature = 1.2: The outputs showed alot more variety. The model introduced new localized vocabulary and concepts in every run, though high temperature also risks occasional minor hallucinations regarding exact local language translations.

A low temperature regime (0.0) is appropriate because a financial decision-support system needs to be consistent, deterministic, and strict. If given identical financial data, the system should always output the exact same risk assessment and decision logic. A higher temperature would introduce random variance, leading to inconsistent loan evaluations for identical applicants.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [5]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [6]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
test_letters = ["L002", "L006"]

for letter_id in test_letters:
    letter_text = LETTERS[letter_id]
    v1_user_prompt = f"Summarize this:\n\n{letter_text}"

    res = ask_llm(v1_user_prompt, temperature=0.7)
    print(f"[{letter_id} - V1 Output]:\n{res.choices[0].message.content.strip()}\n")

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to write a concise 3-4 sentence factual brief summarizing loan applications. "
    "Be completely objective, neutral, and strict. Do NOT invent, assume, or hallucinate any details "
    "not explicitly stated in the text."
)

print("SUMMARY PROMPT V2 (Structured Template)")

for letter_id in test_letters:
    letter_text = LETTERS[letter_id]
    v2_user_prompt = f"Summarize this loan application:\n\n{letter_text}"

    res = ask_llm(
        v2_user_prompt, system_prompt=SUMMARY_SYSTEM_PROMPT_V2, temperature=0.0
    )
    print(f"[{letter_id} - V2 Output]:\n{res.choices[0].message.content.strip()}\n")

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.


[L002 - V1 Output]:
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking a loan of GHS 25,000 to repair his vehicle's engine and settle personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances recover. However, he currently has no collateral to offer.

[L006 - V1 Output]:
Kofi, a 22-year-old, is requesting a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He claims to be "business-minded" based on his friends' opinions, but has no prior experience or collateral to offer. He promises to repay the loan within a year, relying on his trustworthiness as assurance.

SUMMARY PROMPT V2 (Structured Template)
[L002 - V2 Output]:
Kwame Boateng, a commercial driver in Kumasi, has applied for a GHS 25,000 loan to repair his trotro engine and settle personal debts. He cites slow business, but expects an impro

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 
1. 
V1 was vague when explaining applicants' repayment capability. For example, in L002, V1 stated "promises to repay the loan as soon as possible", whereas V2 explicitly highlighted the risk, "He has not specified a repayment schedule, stating only that he can pay back whenever the money comes."
V1 uses informal language and quoted subjective statements as fact ("claims to be 'business-minded' based on his friends' opinions") whereas, V2 converted the text into an objective financial brief focusing on what has and hasn't been actioned, "The applicant has not yet initiated any of these ventures."

2. 
It is essential because microfinance loan officers rely on these summaries to assess any and all financial risk and approve or deny a load. If an LLM invents details that makes an applicant appear as a better candidate by inventing non-existant collateral, inflating revenue or assuming a repayment plan the applicant didn't agree to, it could lead to irresponsible lending and cause the institution to experience financial loss. This failure mode is called Hallucination in the LLM literature.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [7]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

EXTRACT_SYSTEM_PROMPT = """You are a precise data extraction assistant for a microfinance institution.
Your task is to extract specific fields from loan application letters and return only a valid JSON object.

Strict Rules:
1. Return only raw JSON.
2. The JSON object must contain EXACTLY these keys:
   - "applicant_name": string
   - "amount_ghs": number
   - "purpose": string
   - "monthly_profit_ghs": number or null
   - "has_collateral_or_guarantor": boolean
   - "repayment_months": number or null
3. If a field is not explicitly stated in the letter, set its value to null. Do NOT guess, infer, or hallucinate values.

Example
Input Letter:
"My name is Megan Owusu. I run an online vintage fashion thrift store called RetroVault. I currently operate out of a rented warehouse space and earn a steady monthly profit of GHS 3,200. I am applying for a loan of GHS 20,000 to open a in-person store in Osu. I propose a steady repayment plan of GHS 1,000 monthly over 24 months. My warehouse inventory valued at GHS 15,000 will serve as collateral."

Output JSON:
{
  "applicant_name": "Megan Owusu",
  "amount_ghs": 20000,
  "purpose": "open a in-person vintage thrift store",
  "monthly_profit_ghs": 3200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 24
}
"""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    user_prompt = (
        f"Extract structured data from this loan application letter:\n\n{letter_text}"
    )

    try:
        res = ask_llm(user_prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=0.0)
        raw_text = res.choices[0].message.content if hasattr(res, "choices") else res

        cleaned_text = raw_text.strip()
        if cleaned_text.startswith("```"):
            lines = cleaned_text.splitlines()
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].startswith("```"):
                lines = lines[:-1]
            cleaned_text = "\n".join(lines).strip()

        return json.loads(cleaned_text)
    except Exception as e:
        print(f"Warning: Failed to parse JSON response. Error: {e}")
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)
    if extracted:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df_extracted = pd.DataFrame(results)

cols = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]
df_extracted = df_extracted[cols]
df_extracted

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,poultry farm for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. Including test data in few-shot examples causes data leakage. If the model sees one of the evaluation letters in its system prompt, the performance on said letter then becomes a measure of memorization and pattern recognition rather than genuine data extraction.

2. Without that instruction, the model is likely to make inferences when a field is missing

3. Data extraction requires clear, exact and reproducible outputs, when temperature = 0 the model operates using greedy decoding, meaning it always selects the token with the highest probability at every step. This isn't best for creative tasks because they require more diversity and variance.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM_PROMPT = """You are a decision-support assistant for a microfinance loan officer in Ghana. 
Your role is to analyze a loan application alongside its extracted structured data and prepare a balanced, factual recommendation brief.

IMPORTANT RULE:
You must support human decision-making, not replace it. Do not make a final credit decision such as "Approve" or "Reject". 
The final decision is solely made by the human loan officer.

Format your response strictly into these 4 sections:

1. Strengths:
- Bullet points grounding key positive factors (savings history, collateral/guarantor, existing revenue, clear operational goals).

2. Risks / Red Flags:
- Bullet points identifying potential financial, operational, or credit risks.

3. Missing Information:
- Specific documents, records, or clarifications the loan officer should request from the applicant.

4. Suggested Next Step:
- Actionable next steps for the loan officer (e.g., "Invite applicant for an in-person interview", "Request past 6 months of bank statements", "Flag for senior credit review"). Do NOT say approve or reject.
"""


def generate_decision_brief(letter_text, extracted_json):
    user_prompt = f"""
Application Letter:
{letter_text}

Extracted JSON Data:
{json.dumps(extracted_json, indent=2)}

Please generate the loan decision-support brief following the required structure.
"""
    res = ask_llm(user_prompt, system_prompt=BRIEF_SYSTEM_PROMPT, temperature=0.0)
    return (
        res.choices[0].message.content.strip()
        if hasattr(res, "choices")
        else res.strip()
    )


# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted_row = df_extracted[df_extracted["letter_id"] == letter_id].to_dict(
        orient="records"
    )[0]
    briefs[letter_id] = generate_decision_brief(letter_text, extracted_row)

for target_id in ["L001", "L002", "L006"]:
    print(f"DECISION SUPPORT BRIEF FOR {target_id}")
    print(briefs[target_id])
    print("\n")

DECISION SUPPORT BRIEF FOR L001
## 1. Strengths:
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market, indicating stability and familiarity with the market.
* She has a consistent savings history, having saved GHS 2,500 over two years with the susu scheme without missing any contributions, demonstrating her ability to manage finances and commit to regular payments.
* The applicant has a clear operational goal to expand her business into frozen foods, which could potentially increase her profit margin.
* She has a guarantor, her sister, who is a teacher, providing an additional layer of security for the loan.
* Akosua Mensah has proposed a repayment plan of GHS 450 monthly over 20 months, which seems manageable given her current monthly profit of GHS 900.

## 2. Risks / Red Flags:
* The loan amount of GHS 8,000 is significant compared to her savings and monthly profit, which might pose a risk if her business expansion do

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. 
L003
The system effectively identified key financial and operational strengths grounded in the application, including Efua Darko's registered dressmaking business (BN-2019-4482), existing staff (3 apprentices), high proven profit (GHS 2,800 monthly), GCB Bank fixed deposit collateral (GHS 5,000), and 18 months of available sales records. For red flags, the model correctly highlighted manageable, real-world risks such as heavy revenue reliance on seasonal Christmas demand and the operational risks involved in expanding production capacity with new industrial sewing machines.

L006
The system accurately identified major credit red flags while keeping strengths strictly factual and limited to soft traits like applicant enthusiasm, youth, and entrepreneurial ambition. On the risk side, it correctly flagged critical vulnerabilities: requesting a substantial loan (GHS 50,000) for unstarted ventures, attempting to launch three completely unrelated businesses simultaneously (car wash, provision shop, Dubai phone imports), zero prior business experience, absence of collateral or a guarantor, and an entirely speculative 1-year repayment timeline based on personal trust rather than existing cash flow.

2.
Practical Reason: LLMs cannot perform physical site visits, verify original bank documents or bear legal and financial responsibility for loan defaults. If an AI explicitly outputs "Approve" or "Reject", loan officers may blindly accept AI outputs without performing due diligence, exposing the microfinance institution to credit loss and regulatory risk.

Ethical Reason: Credit decisions directly impact individuals' economic livelihoods. Fully automating credit rejections denies applicants empathy and a fair right to appeal. Additionally, it prevents the officer from allowing context and nuance to be considered in the decison. Keeping a Human-in-the-Loop ensures that high-stakes financial decisions impacting real people remain in the hands of actual professionals.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 36bd3c1

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.